In [4]:

import sys
print("Executable:", sys.executable)
print("\nPaths:")
for p in sys.path:
    print(p)

import IMP
import IMP.atom
import IMP.core
import IMP.algebra
import IMP.pmi
import IMP.pmi.topology



Executable: /home/sree/miniforge3/envs/impjax/bin/python

Paths:
/home/sree/miniforge3/envs/impjax/lib/python310.zip
/home/sree/miniforge3/envs/impjax/lib/python3.10
/home/sree/miniforge3/envs/impjax/lib/python3.10/lib-dynload

/home/sree/miniforge3/envs/impjax/lib/python3.10/site-packages


In [5]:
def create_multi_protein_bead_system():
    """
    Create a system with multiple protein types represented as spherical beads
    """
    
    # Step 1: Create the model and system
    model = IMP.Model()
    system = IMP.pmi.topology.System(model, name="MultiProteinSystem")
    
    # Step 2: Create a state (single conformational state)
    state = system.create_state()
    
    # Step 3: Define protein properties
    protein_configs = [
        {
            'name': 'ProteinA',
            'copies': 8,
            'radius': 24.0,  # Angstroms
            'sequence': 'A',  
            'base_coords': IMP.algebra.Vector3D(0, 0, 0),
            'chain_id' : 'A'
        },
        {
            'name': 'ProteinB', 
            'copies': 8,
            'radius': 14.0,
            'sequence': 'AA',  
            'base_coords': IMP.algebra.Vector3D(50, 0, 0),
            'chain_id' : 'B'
        },
        {
            'name': 'ProteinC',
            'copies': 16, 
            'radius': 16.0,
            'sequence': 'L',
            'base_coords': IMP.algebra.Vector3D(100, 0, 0),
            'chain_id' : 'C'
        }
    ]

    # Step 4: Create molecules and their copies
    all_molecules = []
    
    for prot_config in protein_configs:
        # Create the first molecule of this type
        mol = state.create_molecule(
            name=prot_config['name'],
            sequence=prot_config['sequence'],
            chain_id=prot_config['chain_id']
        )
        
        # Add bead representation - use all residues (don't specify residues parameter)
        print(f"Adding representation for {prot_config['name']}") 
        mol.add_representation(
            resolutions=1,  # This creates a single bead per residue
            setup_particles_as_densities=False
        )
        
        all_molecules.append(mol)
        
        # Create copies (clones) of this molecule
        for copy_num in range(1, prot_config['copies']):
            clone = mol.create_clone(
                chain_id=f"{prot_config['chain_id']}{copy_num}"
            )
            all_molecules.append(clone)
    
    # Step 5: Build the system
    system.build()
    
    # Step 6: Set up coordinates and radii for the bead particles
    setup_bead_coordinates(system, protein_configs)
    
    return system, all_molecules

def setup_bead_coordinates(system, protein_configs):
    """
    Set coordinates and radii for bead particles after building
    """
    import random
    
    # Get the hierarchy
    hier = system.get_hierarchy()
    
    # Walk through the hierarchy to find bead particles
    config_idx = 0
    copy_counts = [0, 0, 0]  # Track copies for each protein type
    
    for state in IMP.atom.get_by_type(hier, IMP.atom.STATE_TYPE):
        for molecule in IMP.atom.get_by_type(state, IMP.atom.MOLECULE_TYPE):
            mol_name = molecule.get_name()
            
            # Determine which protein config this molecule belongs to
            if mol_name.startswith('ProteinA'):
                config = protein_configs[0]
                copy_idx = copy_counts[0]
                copy_counts[0] += 1
            elif mol_name.startswith('ProteinB'):
                config = protein_configs[1] 
                copy_idx = copy_counts[1]
                copy_counts[1] += 1
            elif mol_name.startswith('ProteinC'):
                config = protein_configs[2]
                copy_idx = copy_counts[2]
                copy_counts[2] += 1
            else:
                continue
                
            # Find bead particles (fragments) in this molecule
            for fragment in IMP.atom.get_by_type(molecule, IMP.atom.FRAGMENT_TYPE):
                if IMP.core.XYZR.get_is_setup(fragment):
                    # Set up the bead with radius and coordinates
                    bead = IMP.core.XYZR(fragment)
                    bead.set_radius(config['radius'])
                    
                    # Position beads in a grid pattern around base coordinates
                    grid_x = copy_idx % 4  # 4x2 grid for 8 copies, 4x4 for 16
                    grid_y = copy_idx // 4
                    spacing = config['radius'] * 3  # Space them out
                    
                    coord = IMP.algebra.Vector3D(
                        config['base_coords'][0] + grid_x * spacing,
                        config['base_coords'][1] + grid_y * spacing,
                        config['base_coords'][2]
                    )
                    bead.set_coordinates(coord)

def analyze_system(system):
    """
    Print information about the created system
    """
    print("=== System Analysis ===")
    hier = system.get_hierarchy()
    
    print(f"System: {hier.get_name()}")
    print(f"Number of states: {system.get_number_of_states()}")
    
    for state_idx, state in enumerate(system.get_states()):
        print(f"\nState {state_idx}:")
        molecules = state.get_molecules()
        
        for mol_name, mol_list in molecules.items():
            print(f"  Molecule '{mol_name}': {len(mol_list)} copies")
            
            # Print details of first copy
            mol = mol_list[0]
            hier_mol = mol.get_hierarchy()
            
            # Count fragments (beads)
            fragments = IMP.atom.get_by_type(hier_mol, IMP.atom.FRAGMENT_TYPE)
            print(f"    - Fragments/beads: {len(fragments)}")
            
            if fragments:
                frag = fragments[0]
                if IMP.core.XYZR.get_is_setup(frag):
                    bead = IMP.core.XYZR(frag)
                    coord = bead.get_coordinates()
                    radius = bead.get_radius()
                    print(f"    - First bead: center=({coord[0]:.1f}, "
                          f"{coord[1]:.1f}, {coord[2]:.1f}), radius={radius:.1f}")

# Example usage
if __name__ == "__main__":
    # Create the system
    system, molecules = create_multi_protein_bead_system()
    
    # Analyze what we created
    analyze_system(system)
    
    # The system is now ready for further modeling steps like:
    # - Setting up rigid bodies
    # - Adding restraints
    # - Running molecular dynamics or Monte Carlo sampling
    
    print(f"\nTotal molecules created: {len(molecules)}")
    print("System ready for modeling")

Adding representation for ProteinA
Adding representation for ProteinB
Adding representation for ProteinC
done building "ProteinA" Chain A7 Copy: 7
done building "ProteinA" Chain A6 Copy: 6
done building "ProteinA" Chain A5 Copy: 5
done building "ProteinA" Chain A4 Copy: 4
done building "ProteinA" Chain A3 Copy: 3
done building "ProteinA" Chain A2 Copy: 2
done building "ProteinA" Chain A1 Copy: 1
done building "ProteinA" Chain A Copy: 0
done building "ProteinB" Chain B7 Copy: 7
done building "ProteinB" Chain B6 Copy: 6
done building "ProteinB" Chain B5 Copy: 5
done building "ProteinB" Chain B4 Copy: 4
done building "ProteinB" Chain B3 Copy: 3
done building "ProteinB" Chain B2 Copy: 2
done building "ProteinB" Chain B1 Copy: 1
done building "ProteinB" Chain B Copy: 0
done building "ProteinC" Chain C15 Copy: 15
done building "ProteinC" Chain C14 Copy: 14
done building "ProteinC" Chain C13 Copy: 13
done building "ProteinC" Chain C12 Copy: 12
done building "ProteinC" Chain C11 Copy: 11
done 

# Model system of two interacting peptides using IMP-JAX
* derived restraints from the ground truth structure
* Using JAX functionalities within IMP to score this system
* Use the wrapper to deal with calling blackjax samplers

In [42]:
import sys
import IMP
import IMP.core
import IMP.algebra
import IMP.atom
import IMP.container

import IMP.pmi.restraints.crosslinking
import IMP.pmi.restraints.stereochemistry
import IMP.pmi.restraints.em
import IMP.pmi.restraints.basic
import IMP.pmi.tools
import IMP.pmi.samplers
import IMP.pmi.output
import IMP.pmi.macros
import IMP.pmi.topology
import ihm.cross_linkers

import sys
import os

# read in the fasta and pdb 
# Only flexible beads at this point, no rigid bodies 
data_dir = os.path.join(os.getcwd(), "data")
pdb_dir = os.path.join(data_dir, "pdb")
fasta_dir = os.path.join(data_dir, "fasta")

topology_file = os.path.join(data_dir, "topology.txt")

m = IMP.Model()

# Read in topology file
topology = IMP.pmi.topology.TopologyReader(topology_file,
                                           pdb_dir=pdb_dir,
                                           fasta_dir=fasta_dir)

# Sanity check: print parsed components before building
print("Parsed topology components:")
for c in topology.get_components():
    print(f"  mol={c.molname}  pdb={c.pdb_file}  chain={c.chain}  "
          f"res={c.residue_range}  rb={c.rigid_body}")

bs = IMP.pmi.macros.BuildSystem(m, name="ToyModel", resolutions=[1])
bs.add_state(topology)
root_hier, dof = bs.execute_macro()
output = IMP.pmi.output.Output()
output.init_rmf("ini_all.rmf3", [root_hier])
output.write_rmf("ini_all.rmf3")
output.close_rmf("ini_all.rmf3")

output_objects = [] # keep a list of functions that need to be reported
rmf_restraints = []

print("\nBuild complete.")
print("Movers:", dof.get_movers())

# Print molecule/bead counts per type
for state in IMP.atom.get_by_type(root_hier, IMP.atom.STATE_TYPE):
    for mol in IMP.atom.get_by_type(state, IMP.atom.MOLECULE_TYPE):
        leaves = IMP.core.get_leaves(mol)
        print(f"  {mol.get_name()} : {len(leaves)} bead(s)")

molecules = bs.get_molecules()[0]
print("\nMolecules in the system:", molecules)

for mol_names in molecules:
    for mol in molecules[mol_names]:
        cr = IMP.pmi.restraints.stereochemistry.ConnectivityRestraint(mol)
        cr.set_label(mol.get_name())
        cr.add_to_model()
        print(f"Added connectivity restraint for molecule: {mol.get_name()}")
        print(cr)
        output_objects.append(cr)

Parsed topology components:
  mol=KCOIL  pdb=/home/sree/git/jax_bhm/imp_testing/string_of_beads/data/pdb/kcoil_ecoil.pdb  chain=A  res=(1, 21)  rb=1
  mol=KCOIL  pdb=BEADS  chain=None  res=(22, 31)  rb=None
  mol=KCOIL  pdb=/home/sree/git/jax_bhm/imp_testing/string_of_beads/data/pdb/kcoil_ecoil.pdb  chain=A  res=(32, 52)  rb=2
  mol=ECOIL  pdb=/home/sree/git/jax_bhm/imp_testing/string_of_beads/data/pdb/kcoil_ecoil.pdb  chain=B  res=(1, 21)  rb=3
  mol=ECOIL  pdb=BEADS  chain=None  res=(22, 31)  rb=None
  mol=ECOIL  pdb=/home/sree/git/jax_bhm/imp_testing/string_of_beads/data/pdb/kcoil_ecoil.pdb  chain=B  res=(32, 52)  rb=4
BuildSystem.add_state: setting up molecule KCOIL copy number 0
BuildSystem.add_state: molecule KCOIL sequence has 52 residues
BuildSystem.add_state: ---- setting up domain 0 of molecule KCOIL
BuildSystem.add_state: -------- domain 0 of molecule KCOIL extends from residue 1 to residue 21 
BuildSystem.add_state: -------- domain 0 of molecule KCOIL represented by pdb fil